In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

import sys
from pathlib import Path

sys.path.append(str(Path("../src").resolve()))   # folder that CONTAINS the .py file

import utils
from utils import clean, get_data, get_results, split, model_input_features       # for src/my_utils.py

## where trained models are saved (project-root models/ folder)
MODEL_DIR = Path("../models").resolve()
MODEL_DIR.mkdir(exist_ok=True)

In [2]:


# make 2026 consistent with 2024/2025: drop the 2026-only columns and use the same column order
# (2026 lacks the defensive columns, which are ~all NaN in 2025 and unused by the position frames -> filled with NaN)
df = pd.read_csv("../data/df_main_final.csv")
df = model_input_features(df)


/tmp/ipykernel_13415/3326549339.py:3: DtypeWarning: Columns (0: season_type) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/df_main_final.csv")
/home/tristan/Documents/MSDS/CSCI 5502/CSCI_5502_Fantasy_Project/src/utils.py:100: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_last3'] = (
/home/tristan/Documents/MSDS/CSCI 5502/CSCI_5502_Fantasy_Project/src/utils.py:100: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_last3'] = (
/home/tristan/Documents/MSDS/CS

In [3]:
df.head()


,player_id,player,position,team,season,week,eventId,date,opponent,atVs,...,passingTouchdowns_cum_career_avg,passingTouchdowns_season_avg,interceptions_cum_career_avg,interceptions_season_avg,fumblesLost_cum_career_avg,fumblesLost_season_avg,fantasy_points_last3,receivingTargets_last3,rushingAttempts_last3,passingAttempts_last3
0,5529,Philip Rivers,QB,IND,2025,15,401772799,2025-12-14 21:25:00+00:00,SEA,away,...,1.000000,1.000000,1.0,1.0,NaN,NaN,6.800000,NaN,1.000000,27.000000
1,5529,Philip Rivers,QB,IND,2025,16,401772824,2025-12-23 01:15:00+00:00,SF,home,...,1.500000,1.500000,1.0,1.0,NaN,NaN,11.940000,NaN,0.500000,31.000000
2,5529,Philip Rivers,QB,IND,2025,17,401772915,2025-12-28 18:00:00+00:00,JAX,home,...,1.333333,1.333333,1.0,1.0,NaN,NaN,10.553333,NaN,0.666667,30.666667
3,8439,Aaron Rodgers,QB,NYJ,2024,1,401671696,2024-09-10 00:15:00+00:00,SF,away,...,1.000000,1.000000,1.0,1.0,NaN,NaN,8.580000,NaN,1.000000,21.000000
4,8439,Aaron Rodgers,QB,NYJ,2024,2,401671636,2024-09-15 17:00:00+00:00,TEN,away,...,1.500000,1.500000,0.5,0.5,NaN,NaN,11.860000,NaN,1.500000,25.500000


## Build the same modeling frame as `XGBOOST.ipynb`
The clustered models use exactly the same rows, target, features and train/val/test split as the global model.
The only difference is that the rows are partitioned by position group and one model is fit per group.

In [4]:
# #Sort chronologically by player (required before the expanding/rolling features below)
# df = df.sort_values(['player_id', 'season', 'week']).reset_index(drop=True)

# #Create target (Next week's fantasy score) and next week's context via shift(-1)
# #group by player AND season so a season's last game doesn't take next season's week 1 as its target
# g = df.groupby(['player_id', 'season'])
# df['target_next_week_fantasy'] = g['fantasy_points'].shift(-1)
# df['next_week_opponent'] = g['opponent'].shift(-1)
# df['next_week_atVs'] = g['atVs'].shift(-1)
# df['next_week_projected'] = g['projected_points'].shift(-1)   # ESPN's projection for the game being predicted (baseline only)
# for col in ['roof', 'surface', 'temp', 'wind', 'location']:
#     df[f'next_week_{col}'] = g[col].shift(-1)
# next_week_opp_def_rank / team ranks not created: they are season-end values, so they leak future games

In [5]:
# #Compute cumulative career and season-to-date stats up to current row (Week N)
# stats_to_accumulate = [
#     'fantasy_points', 'receptions', 'receivingTargets', 'receivingYards', 
#     'receivingTouchdowns', 'rushingAttempts', 'rushingYards', 'rushingTouchdowns',
#     'completions', 'passingAttempts', 'passingYards', 'passingTouchdowns', 
#     'interceptions', 'fumblesLost'
# ]

# for col in stats_to_accumulate:
#     # 1. Career cumulative mean (df is sorted by player_id, so .values lines up with the rows)
#     df[f'{col}_cum_career_avg'] = (
#         df.groupby('player_id')[col]
#         .expanding()
#         .mean()
#         .values
#     )
    
#     # 2. Season-to-date cumulative mean
#     df[f'{col}_season_avg'] = (
#         df.groupby(['player_id', 'season'])[col]
#         .expanding()
#         .mean()
#         .values
#     )

# for col in ['fantasy_points', 'receivingTargets', 'rushingAttempts', 'passingAttempts']:
#     df[f'{col}_last3'] = (
#         df.groupby(['player_id', 'season'])[col]
#         .transform(lambda s: s.rolling(3, min_periods=1).mean()))

In [6]:
# ## encode categorical columns (same as the global model; cast on the full frame so every group shares one vocabulary)
# categorical_cols = ['player','position', 'next_week_opponent', 'next_week_atVs', 'next_week_surface',
#                     'next_week_roof', 'next_week_location']
# for col in categorical_cols:
#     df[col] = df[col].astype('category')

# target = 'target_next_week_fantasy'

# ## same excluded columns as the global model
# irrelevant_columns = [
#     'Unnamed: 0', 'player_id', 'team', 'opponent', 'date',
#     'eventId', 'game_id', 'gameday', 'gametime', 'stadium', 'result',
#     'score', 'fantasy_points', 'target_next_week_fantasy', 'espn',
#     '_set', 'season',
#     'med_proj_pts',
#     'projected_points',
#     'atVs', 'roof', 'surface', 'temp', 'wind', 'location',
#     'opp_def_rank', 'team_pass_rank', 'team_rush_rank', 'next_week_projected' 
# ]
# relevant_columns = [col for col in df.columns if col not in irrelevant_columns]
# irrelevant = [col for col in df.columns if col not in relevant_columns]   # used by get_data
# print(len(relevant_columns), 'features')



In [7]:
# #remove invalid players that model does not see
# valid_players = df[df["season"] != 2026]["player"].unique().tolist()
# # df = df.loc[(~df["player"].isin(players))]
# df = df.loc[df["player"].isin(valid_players)]
# df.shape


In [8]:
## position groups from Modeling_data_prep.ipynb (index is kept so get_results can look rows up in these frames)
groups = {'RB': ['RB'], 'QB_TE': ['QB', 'TE'], 'WR_K': ['WR', 'K']}
rb_df, qb_te, wr_k = (df[df['position'].isin(pos)] for pos in groups.values())

In [9]:
rb_df.head()

,player_id,player,position,team,season,week,eventId,date,opponent,atVs,...,passingTouchdowns_cum_career_avg,passingTouchdowns_season_avg,interceptions_cum_career_avg,interceptions_season_avg,fumblesLost_cum_career_avg,fumblesLost_season_avg,fantasy_points_last3,receivingTargets_last3,rushingAttempts_last3,passingAttempts_last3
347,15807,Cordarrelle Patterson,RB,PIT,2024,1,401671744,2024-09-08 17:00:00+00:00,ATL,away,...,NaN,NaN,NaN,NaN,0.0,0.0,1.300000,0.000000,4.000000,NaN
348,15807,Cordarrelle Patterson,RB,PIT,2024,2,401671668,2024-09-15 20:25:00+00:00,DEN,away,...,NaN,NaN,NaN,NaN,0.0,0.0,0.800000,0.000000,3.000000,NaN
349,15807,Cordarrelle Patterson,RB,PIT,2024,3,401671632,2024-09-22 17:00:00+00:00,LAC,home,...,NaN,NaN,NaN,NaN,0.0,0.0,3.133333,1.666667,3.333333,NaN
350,15807,Cordarrelle Patterson,RB,PIT,2024,4,401671867,2024-09-29 17:00:00+00:00,IND,away,...,NaN,NaN,NaN,NaN,0.0,0.0,5.433333,2.333333,4.000000,NaN
351,15807,Cordarrelle Patterson,RB,PIT,2024,10,401671641,2024-11-10 18:00:00+00:00,WSH,away,...,NaN,NaN,NaN,NaN,0.0,0.0,5.800000,2.333333,4.333333,NaN


In [10]:
# same split as the global model
# train: 2024 + 2025 weeks 1-9, validation: 2025 weeks 10-18, test: 2026 weeks 1-10
# def split(d):
#     train = d[(d['season'] == 2024) | ((d['season'] == 2025) & (d['week'] < 10))]
#     val = d[(d['season'] == 2025) & (d['week'].between(10, 18))]
#     test = d[(d['season'] == 2026) & (d['week'] < 11)]
#     return train, val, test

train_data_1, val_1, test_1 = split(rb_df)
train_data_2, val_2, test_2 = split(qb_te)
train_data_3, val_3, test_3 = split(wr_k)

In [11]:
## Running back only
X1_train, y1_train = get_data(train_data_1, irrelevant, target)
X1_val, y1_val = get_data(val_1, irrelevant, target)
X1_test, y1_test = get_data(test_1, irrelevant, target)
##quarter back/ tight end
X2_train, y2_train = get_data(train_data_2, irrelevant, target)
X2_val, y2_val = get_data(val_2, irrelevant, target)
X2_test, y2_test = get_data(test_2, irrelevant, target)
## Wide reciever/kicker
X3_train, y3_train = get_data(train_data_3, irrelevant, target)
X3_val, y3_val = get_data(val_3, irrelevant, target)
X3_test, y3_test = get_data(test_3, irrelevant, target)

# sanity check: the three groups together are exactly the global model's rows (10,555 / 3,183 / 3,250)
print('train', len(X1_train) + len(X2_train) + len(X3_train),
      'val', len(X1_val) + len(X2_val) + len(X3_val),
      'test', len(X1_test) + len(X2_test) + len(X3_test))
assert list(X1_train.columns) == list(X2_train.columns) == list(X3_train.columns) == relevant_columns

NameError: name 'irrelevant' is not defined

In [68]:
model1_path = MODEL_DIR / "xgb_rb.json"   # RB model, saved after training
model1 = xgb.XGBRegressor(
    tree_method = "hist",
    n_estimators=10000, #Higher is better
    learning_rate=0.005, #lower is better
    max_depth=5, #higher is better
    min_child_weight=3, #mid:1-10
    subsample=0.7, #mid:0-1
    colsample_bytree=0.7, #mid:0-1
    random_state=12, #seed
    enable_categorical=True,
    early_stopping_rounds=50      # stops when validation stops improving (old run overfit after ~50 trees)
)



In [69]:
model1.fit(
    X1_train, y1_train,
    eval_set=[(X1_val, y1_val)],
    verbose=50 #how often to print validation score 
)
model1.save_model(str(model1_path))
print(f"saved model1 to {model1_path}")

[0]	validation_0-rmse:8.08516
[50]	validation_0-rmse:7.34233
[100]	validation_0-rmse:6.86269
[150]	validation_0-rmse:6.56364
[200]	validation_0-rmse:6.37259
[250]	validation_0-rmse:6.25409
[300]	validation_0-rmse:6.19517
[350]	validation_0-rmse:6.16436
[400]	validation_0-rmse:6.14610
[450]	validation_0-rmse:6.14935
[478]	validation_0-rmse:6.14955
saved model1 to /home/tristan/Documents/MSDS/CSCI 5502/CSCI_5502_Fantasy_Project/models/xgb_rb.json


In [70]:
model2_path = MODEL_DIR / "xgb_qb_te.json"   # QB/TE model, saved after training
model2 = xgb.XGBRegressor(
    tree_method = "hist",
    n_estimators=10000, #Higher is better
    learning_rate=0.005, #lower is better
    max_depth=5, #higher is better
    min_child_weight=3, #mid:1-10
    subsample=0.7, #mid:0-1
    colsample_bytree=0.7, #mid:0-1
    random_state=12, #seed
    enable_categorical=True,
    early_stopping_rounds=50      # stops when validation stops improving (old run overfit after ~50 trees)
)
model2.fit(X2_train, y2_train,
        eval_set=[(X2_val, y2_val)],
        verbose=50)
model2.save_model(str(model2_path))
print(f"saved model2 to {model2_path}")

[0]	validation_0-rmse:7.98787
[50]	validation_0-rmse:7.15576
[100]	validation_0-rmse:6.62233
[150]	validation_0-rmse:6.29042
[200]	validation_0-rmse:6.10090
[250]	validation_0-rmse:5.98736
[300]	validation_0-rmse:5.93597
[350]	validation_0-rmse:5.91535
[400]	validation_0-rmse:5.91412
[437]	validation_0-rmse:5.91836
saved model2 to /home/tristan/Documents/MSDS/CSCI 5502/CSCI_5502_Fantasy_Project/models/xgb_qb_te.json


In [71]:
model3_path = MODEL_DIR / "xgb_wr_k.json"   # WR/K model, saved after training
model3 = xgb.XGBRegressor(
    tree_method = "hist",
    n_estimators=10000, #Higher is better
    learning_rate=0.005, #lower is better
    max_depth=5, #higher is better
    min_child_weight=3, #mid:1-10
    subsample=0.7, #mid:0-1
    colsample_bytree=0.7, #mid:0-1
    random_state=12, #seed
    enable_categorical=True,
    early_stopping_rounds=50      # stops when validation stops improving (old run overfit after ~50 trees)
)
model3.fit(X3_train, y3_train,
        eval_set=[(X3_val, y3_val)],
        verbose=50)
model3.save_model(str(model3_path))
print(f"saved model3 to {model3_path}")

[0]	validation_0-rmse:6.97319
[50]	validation_0-rmse:6.45844
[100]	validation_0-rmse:6.14786
[150]	validation_0-rmse:5.96870
[200]	validation_0-rmse:5.87285
[250]	validation_0-rmse:5.82331
[300]	validation_0-rmse:5.80624
[350]	validation_0-rmse:5.80784
[375]	validation_0-rmse:5.80915
saved model3 to /home/tristan/Documents/MSDS/CSCI 5502/CSCI_5502_Fantasy_Project/models/xgb_wr_k.json


In [72]:
## get our results for model 1
rb_results = get_results(rb_df, X1_test, y1_test, model1)

Model: MAE=4.499  RMSE=6.312  R2=0.429
ESPN: MAE=3.991  RMSE=5.750  R2=0.526


In [73]:
# model 2 results
qb_te_results=get_results(qb_te, X2_test, y2_test, model2)

Model: MAE=4.700  RMSE=6.211  R2=0.456
ESPN: MAE=3.894  RMSE=5.584  R2=0.561


In [74]:
wr_k_results = get_results(wr_k, X3_test, y3_test, model3)

Model: MAE=4.671  RMSE=6.245  R2=0.320
ESPN: MAE=4.148  RMSE=5.733  R2=0.427


In [75]:
all_pos = pd.concat([rb_results, qb_te_results, wr_k_results], ignore_index=True)
position_table = pd.crosstab(all_pos['position'], all_pos['winner'], margins=True)
position_table

winner,Model,espn,All
position,,,
K,125,109,234
QB,105,199,304
RB,302,418,720
TE,259,442,701
WR,416,653,1069
All,1207,1821,3028


## Player lookup
Enter a player's name to compare the model's predictions with their actual fantasy points and ESPN's projection for each test week (2026). `predicted_week` is the player's next game after the week the features come from (bye weeks are skipped).

In [18]:
# def next_game_weeks(player, season):
#     """Map each week a player played to the week of their next game (skips bye weeks)."""
#     weeks = sorted(df.loc[(df["player"] == player) & (df["season"] == season), "week"].unique())
#     return dict(zip(weeks, weeks[1:]))

# def player_report(name, results=all_pos, plot=True):
#     """Show model vs ESPN vs actual for one player (case-insensitive, partial names OK)."""
#     matches = results[results["player"].str.contains(name, case=False, na=False, regex=False)]
#     if matches.empty:
#         print(f"No test rows found for '{name}'.")
#         return None
#     players = matches["player"].unique()
#     if len(players) > 1:
#         print(f"Multiple players match '{name}': {list(players)}. Be more specific.")
#         return None

#     p = matches.sort_values(["season", "week"]).copy()
#     p["predicted_week"] = [next_game_weeks(players[0], s).get(w) for s, w in zip(p["season"], p["week"])]
#     cols = ["player", "position", "season", "predicted_week", "next_week_opponent",
#             "fantasy_points", "model_prediction", "espn_projected",
#             "model_error_abs", "espn_error_abs", "winner"]
#     table = p[cols].rename(columns={"fantasy_points": "actual", "next_week_opponent": "opponent"}).round(2)

#     print(f"{players[0]} ({p['position'].iloc[0]}): {len(p)} test weeks")
#     print(f"  Model MAE: {p['model_error_abs'].mean():.2f}   ESPN MAE: {p['espn_error_abs'].mean():.2f}")
#     print(f"  Model wins {(p['winner'] == 'Model').sum()} / {len(p)} weeks")

#     if plot:
#         fig, ax = plt.subplots(figsize=(9, 4))
#         ax.plot(table["predicted_week"], table["actual"], marker="o", label="Actual")
#         ax.plot(table["predicted_week"], table["model_prediction"], marker="s", label="Model")
#         ax.plot(table["predicted_week"], table["espn_projected"], marker="^", linestyle="--", label="ESPN")
#         ax.set_xlabel("Week")
#         ax.set_ylabel("Fantasy points")
#         ax.set_title(f"{players[0]}: predicted vs actual")
#         ax.legend()
#         plt.show()
#     return table.reset_index(drop=True)

# player_report("Christian McCaffrey")

## Modified Groups

Based on the findings in the ```Modeling_data_prep.ipynb``` we found the clusters with the most similar coefficient vectors. However, the cluster of Wide Reciever and Kicker makes no sense in terms of the game of football and based on our Spearman coefficients, Wide reciever was also close to QB. I would like to explore the potential cluster of WR, TE, WR and leave RB and K in their own seperate model.

In [19]:
# #sort values to apply shifts
# df1 = df.sort_values(['player_id', 'season', 'week']).reset_index(drop=True)
# # shift data
# g = df1.groupby(['player_id', "season"])
# df1["target_fantasy_points"] = g["fantasy_points"].shift(-1)
# df1['next_week_opponent'] = g['opponent'].shift(-1)
# df1['next_week_atVs'] = g['atVs'].shift(-1)
# df1['next_week_projected'] = g['projected_points'].shift(-1)

# # drop the last week of each season (no next week -> NaN targets, already seen by the previous week's shift)
# df1 = df1[df1["week"] < df1.groupby("season")["week"].transform("max")].reset_index(drop=True)

In [20]:


#already have model for running back so we are going to leace that alone
# qb, wr, te, k = df1[df1["position"] == "QB"], df1[df1["position"] == "WR"], df1[df1["position"] == "TE"], df1[df1["position"] == "K"]

# df2 = pd.concat([qb, wr, te])

# df2[shared_cat] = df2[shared_cat].astype("category")
# k[shared_cat] = k[shared_cat].astype("cateogory")

# training1 = df2[(df2["season"] == 2024) | ((df2["season"] == 2025) & (df2["week"] < 10))]
# val1 = df2[(df2["season"] == 2025) & (df2["week"] >= 10)]
# test1 = df2[df2["season"] == 2026]

# k_train = k[(k["season"] == 2024)|((k["season"]==2025) & (k["week"] < 10))]
# k_val = k[(k["season"] == 2025) & (k["week"] >= 10)]
# k_test = k[k["season"] == 2026]

# train_x, train_y = get_data(training1, irrelevant, target)
# val_x, val_y = get_data(val1, irrelevant, target)
# test_x, test_y = get_data(test1, irrelevant, target)

# train_k_x, train_k_y = get_data(k_train, irrelevant, target)
# val_k_x, val_k_y = get_data(k_val, irrelevant, target)
# test_k_x, test_k_y = get_data(k_test, irrelevant, target)

# # df2.head()

